In [19]:
# IMPORT
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.base import clone
import seaborn as sns
import pandas as pd
import numpy as np
import random

from jmetal.operator.mutation import PolynomialMutation
from jmetal.operator.crossover import SBXCrossover
from jmetal.algorithm.multiobjective import NSGAII, SPEA2
from jmetal.core.solution import FloatSolution
from jmetal.util.termination_criterion import StoppingByEvaluations
from jmetal.util.solution import get_non_dominated_solutions
from jmetal.core.problem import Problem


In [20]:
# Lecture des fichiers
df_diabetes = pd.read_csv("diabetes.csv")
df_yeast     = pd.read_csv("yeast.csv")

print("Diabetes shape:", df_diabetes.shape)
print("Yeast shape:", df_yeast.shape)


Diabetes shape: (768, 9)
Yeast shape: (1484, 9)


In [21]:
def evaluate_individual(X, y_true, chromosomes):
    """
    Transforme un chromosome [bi1, bs1, bi2, bs2, ...] en une règle,
    fait la prédiction sur X, et renvoie (precision, recall).
    (F1 est gardée pour analyse, pas pour le front de Pareto.)
    """
    n_attributes = X.shape[1]
    y_pred = []

    for instance in X:
        is_positive = True
        for i in range(n_attributes):
            bi = chromosomes[2 * i]
            bs = chromosomes[2 * i + 1]
            # attribut actif si bi <= bs
            if bi <= bs:
                if not (bi <= instance[i] <= bs):
                    is_positive = False
                    break
        y_pred.append(1 if is_positive else 0)

    y_pred = np.array(y_pred)

    if sum(y_pred) == 0:
        precision = 0.0
        recall    = 0.0
        f1        = 0.0
    else:
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall    = recall_score(y_true, y_pred, zero_division=0)
        f1        = f1_score(y_true, y_pred, average='binary', zero_division=0)

    return precision, recall, f1  # tuple (precision, recall, f1)


In [30]:
class PartialClassifProblemMO(Problem):
    def __init__(self, X_train, y_train, X_test, y_test):
        super().__init__()
        self.X_train = np.array(X_train, dtype=float)
        self.y_train = np.array(y_train, dtype=int)
        self.X_test  = np.array(X_test,  dtype=float)
        self.y_test  = np.array(y_test,  dtype=int)

        self.n_attributes = self.X_train.shape[1]

        # Bornes de chaque attribut
        mins = X_train.min(axis=0)
        maxs = X_train.max(axis=0)
        self.lower_bound = []
        self.upper_bound = []

        for i in range(self.n_attributes):
            self.lower_bound.extend([mins[i], mins[i]])
            self.upper_bound.extend([maxs[i], maxs[i]])

    def number_of_variables(self) -> int:
        return 2 * self.n_attributes

    def number_of_objectives(self) -> int:
        return 2  # precision, recall

    def number_of_constraints(self) -> int:
        return 0

    def evaluate(self, solution: FloatSolution) -> FloatSolution:
        # Évaluer sur X_train
        prec, rec, _ = evaluate_individual(
            X=self.X_train,
            y_true=self.y_train,
            chromosomes=solution.variables
        )

        solution.objectives[0] = -prec
        solution.objectives[1] = -rec

        return solution

    def create_solution(self) -> FloatSolution:
        solution = FloatSolution(
            lower_bound=self.lower_bound,
            upper_bound=self.upper_bound,
            number_of_objectives=self.number_of_objectives(),
            number_of_constraints=self.number_of_constraints()
        )
        solution.variables = [
            random.uniform(self.lower_bound[i], self.upper_bound[i])
            for i in range(self.number_of_variables())
        ]
        return solution

    def name(self) -> str:
        return "PartialClassifProblemMO"

    def get_rule(self, solution: FloatSolution, feature_names=None) -> dict:
        """
        Renvoie une règle "lisible" de la forme :
        { "attribute_name": [min, max] } ou None si l'attribut est inactif.
        """
        n_attributes = self.n_attributes
        rule = {}
        if feature_names is None:
            feature_names = [f"X{i}" for i in range(n_attributes)]

        for i in range(n_attributes):
            bi = solution.variables[2 * i]
            bs = solution.variables[2 * i + 1]
            if bi <= bs:
                rule[feature_names[i]] = [bi, bs]
            else:
                rule[feature_names[i]] = None  # attribut inactif

        return rule

    def evaluate_test(self, solution: FloatSolution):
        """
        Évalue une solution sur le test (X_test, y_test) et renvoie:
        precision, recall, f1, accuracy
        """
        # Adapter chromosomes à X_test
        y_pred = []

        for instance in self.X_test:
            is_positive = True
            for i in range(self.n_attributes):
                bi = solution.variables[2 * i]
                bs = solution.variables[2 * i + 1]
                if bi <= bs:
                    if not (bi <= instance[i] <= bs):
                        is_positive = False
                        break
            y_pred.append(1 if is_positive else 0)

        y_pred = np.array(y_pred)

        if sum(y_pred) == 0:
            precision = 0.0
            recall    = 0.0
            f1        = 0.0
            accuracy  = 0.0
        else:
            precision = precision_score(self.y_test, y_pred, zero_division=0)
            recall    = recall_score(self.y_test, y_pred, zero_division=0)
            f1        = f1_score(self.y_test, y_pred, average='binary', zero_division=0)
            accuracy  = accuracy_score(self.y_test, y_pred)

        return precision, recall, f1, accuracy


In [24]:
# Nettoyage des zéros impossibles (diabète)
zero_impossible_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_diabetes[zero_impossible_cols] = df_diabetes[zero_impossible_cols].replace(0, np.nan)
df_diabetes.fillna(df_diabetes.median(numeric_only=True), inplace=True)

# X, y pour le diabète (classification binaire déjà existante : 0/1)
X_diabetes = df_diabetes.drop('Outcome', axis=1)  # 8 attributs
y_diabetes = df_diabetes['Outcome']                # 0 ou 1

# Split train / test
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(
    X_diabetes,
    y_diabetes,
    test_size=0.2,
    random_state=42,
    stratify=y_diabetes
)

print("X_train_diabetes shape:", X_train_diabetes.shape)
print("y_train_diabetes shape:", y_train_diabetes.shape)
print("Classes dans y_train : 0 =", (y_train_diabetes == 0).sum(), ", 1 =", (y_train_diabetes == 1).sum())


X_train_diabetes shape: (614, 8)
y_train_diabetes shape: (614,)
Classes dans y_train : 0 = 400 , 1 = 214


In [25]:
# Standardisation pour SVM
scaler = StandardScaler()
X_train_diabetes_scaled = scaler.fit_transform(X_train_diabetes)
X_test_diabetes_scaled  = scaler.transform(X_test_diabetes)

# Modèles
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)

c45_model = DecisionTreeClassifier(
    criterion='entropy',
    splitter='best',
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

# Configurations
model_configs = {
    "Random Forest": (rf_model, X_train_diabetes, X_test_diabetes),
    "SVM":         (svm_model, X_train_diabetes_scaled, X_test_diabetes_scaled),
    "C4.5":        (c45_model, X_train_diabetes, X_test_diabetes),
}

# Évaluation
results_diabetes = {}

for name, (model, X_tr, X_te) in model_configs.items():
    model.fit(X_tr, y_train_diabetes)

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]  # proba de la classe 1

    # Métriques
    acc = accuracy_score(y_test_diabetes, y_pred)
    cv_scores = cross_val_score(model, X_tr, y_train_diabetes, cv=5, scoring='accuracy')

    results_diabetes[name] = {
        "Accuracy":   acc,
        "ROC AUC":    roc_auc_score(y_test_diabetes, y_prob),
        "CV Mean":    cv_scores.mean(),
        "CV Std":     cv_scores.std(),
        "y_pred":     y_pred,
        "y_prob":     y_prob,
        "model":      model,
    }

# Affichage des résultats
for name, scores in results_diabetes.items():
    print(f"{name:<12} | Acc: {scores['Accuracy']:.3f} | ROC AUC: {scores['ROC AUC']:.3f} | CV Mean: {scores['CV Mean']:.3f} ± {scores['CV Std']:.3f}")


Random Forest | Acc: 0.779 | ROC AUC: 0.819 | CV Mean: 0.772 ± 0.033
SVM          | Acc: 0.740 | ROC AUC: 0.796 | CV Mean: 0.769 ± 0.018
C4.5         | Acc: 0.675 | ROC AUC: 0.639 | CV Mean: 0.694 ± 0.036


In [26]:
def run_algorithm(algorithm_class, problem, population_size, max_evaluations, crossover_prob=0.9, mutation_prob=None):
    n_vars = problem.number_of_variables()
    if mutation_prob is None:
        mutation_prob = 1.0 / n_vars

    crossover = SBXCrossover(probability=crossover_prob, distribution_index=20)
    mutation = PolynomialMutation(probability=mutation_prob, distribution_index=20)

    # gestion de la compatibilité version jmetalpy
    try:
        termination = StoppingByEvaluations(max_evaluations=max_evaluations)
    except TypeError:
        termination = StoppingByEvaluations(max=max_evaluations)

    algorithm = algorithm_class(
        problem=problem,
        population_size=population_size,
        offspring_population_size=population_size,
        mutation=mutation,
        crossover=crossover,
        termination_criterion=termination
    )

    print(f"Running {algorithm.get_name()} — {max_evaluations} evaluations...")
    algorithm.run()

    try:
        raw_results = algorithm.result()
    except AttributeError:
        raw_results = algorithm.get_result()

    pareto_front = get_non_dominated_solutions(raw_results)
    print(f"Done. Pareto front size: {len(pareto_front)}")
    return pareto_front, algorithm


In [29]:
# Pour pima‑diabète
y_train_bin = (y_train_diabetes == 1).astype(int)
y_test_bin  = (y_test_diabetes  == 1).astype(int)

problem = PartialClassifProblemMO(
    X_train=X_train_diabetes.values,
    y_train=y_train_bin,
    X_test=X_test_diabetes.values,
    y_test=y_test_bin
)

# Lancer NSGAII via run_algorithm
pareto_front, algo = run_algorithm(
    algorithm_class=NSGAII,
    problem=problem,
    population_size=50,
    max_evaluations=5000
)

# Prendre une solution du front (ex: la meilleure en F1)
if pareto_front:
    best_solution = pareto_front[0]   # ou sélection via HV / F1
    rule = problem.get_rule(best_solution, feature_names=X_train_diabetes.columns.tolist())
    print("Règle trouvée :", rule)

    # évaluer sur le test
    prec, rec, f1, acc = problem.evaluate_test(best_solution)
    print(f"Test: prec={prec:.3f}, rec={rec:.3f}, f1={f1:.3f}, acc={acc:.3f}")


[2026-03-24 02:18:49,960] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:18:49,960] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:18:50,037] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:18:50,037] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met


Running NSGAII — 5000 evaluations...


[2026-03-24 02:19:02,053] [jmetal.core.algorithm] [DEBUG] Finished!


Done. Pareto front size: 41
Règle trouvée : {'Pregnancies': None, 'Glucose': None, 'BloodPressure': None, 'SkinThickness': None, 'Insulin': None, 'BMI': None, 'DiabetesPedigreeFunction': None, 'Age': None}
Test: prec=0.351, rec=1.000, f1=0.519, acc=0.351


In [33]:
# 5.1. Paramètres fixes
# ----------------------
N_RUNS = 20                             # ≥ 20 runs par AG
MAX_EVALUATIONS = 100                  # évaluations (fixe pour tous les tests)

# Jeux de données à tester
datasets = {
    "Pima Diabetes": {
        "X_train": X_train_diabetes.values,
        "y_train": (y_train_diabetes == 1).astype(int),
        "X_test":  X_test_diabetes.values,
        "y_test":  (y_test_diabetes  == 1).astype(int),
    },
}


In [34]:
# 3 tailles de population, comme demandé
POP_SIZES = [20, 50, 100]

# Stockage pour le rapport : 2 AG × 2 datasets × 3 tailles × 20 runs
results_mo = {ds_name: {"NSGAII": {}, "SPEA2": {}} for ds_name in datasets}

for dataset_name, data_dict in datasets.items():
    print(f"\n\n=== JEUX DE DONNÉES : {dataset_name} ===")

    X_tr = data_dict["X_train"]
    y_tr = data_dict["y_train"]
    X_te = data_dict["X_test"]
    y_te = data_dict["y_test"]

    for algo_name, algo_class in [("NSGAII", NSGAII), ("SPEA2", SPEA2)]:
        print(f"\n{algo_name} | {dataset_name}")

        results_mo[dataset_name][algo_name] = {}

        for pop_size in POP_SIZES:
            print(f"  Population size = {pop_size} (20 runs)")

            runs = []

            for run_idx in range(N_RUNS):
                # Créer un problème pour ce run (encapsule X_train, y_train, X_test, y_test)
                problem = PartialClassifProblemMO(
                    X_train=X_tr,
                    y_train=y_tr,
                    X_test=X_te,
                    y_test=y_te
                )

                pareto_front_train, _ = run_algorithm(
                    algorithm_class=algo_class,
                    problem=problem,
                    population_size=pop_size,
                    max_evaluations=MAX_EVALUATIONS,
                    crossover_prob=0.9
                )

                # Garder le front de Pareto NON DOMINÉES (sur TRAIN)
                runs.append({
                    "run_idx": run_idx,
                    "pop_size": pop_size,
                    "pareto_train": pareto_front_train,  # fronts de Pareto sur le train
                    # Optionnel : on peut stocker aussi des statistiques rapidement calculées :
                    "precisions": [sol.objectives[0] * (-1) for sol in pareto_front_train],
                    "recalls":    [sol.objectives[1] * (-1) for sol in pareto_front_train]
                })

            results_mo[dataset_name][algo_name][pop_size] = runs

print("Fin du protocole expérimental (tuning + runs).")


[2026-03-24 02:29:17,040] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:17,041] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:29:17,064] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:17,064] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-03-24 02:29:17,197] [jmetal.core.algorithm] [DEBUG] Finished!
[2026-03-24 02:29:17,198] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:17,198] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:29:17,227] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:17,228] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met




=== JEUX DE DONNÉES : Pima Diabetes ===

NSGAII | Pima Diabetes
  Population size = 20 (20 runs)
Running NSGAII — 100 evaluations...
Done. Pareto front size: 8
Running NSGAII — 100 evaluations...


[2026-03-24 02:29:17,366] [jmetal.core.algorithm] [DEBUG] Finished!
[2026-03-24 02:29:17,367] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:17,367] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:29:17,385] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:17,386] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-03-24 02:29:17,525] [jmetal.core.algorithm] [DEBUG] Finished!
[2026-03-24 02:29:17,526] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:17,526] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:29:17,546] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:17,547] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met


Done. Pareto front size: 6
Running NSGAII — 100 evaluations...
Done. Pareto front size: 11
Running NSGAII — 100 evaluations...


[2026-03-24 02:29:17,694] [jmetal.core.algorithm] [DEBUG] Finished!
[2026-03-24 02:29:17,694] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:17,695] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:29:17,719] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:17,719] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-03-24 02:29:17,858] [jmetal.core.algorithm] [DEBUG] Finished!
[2026-03-24 02:29:17,859] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:17,859] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:29:17,871] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:17,872] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met


...

Done. Pareto front size: 8
Running SPEA2 — 100 evaluations...
Done. Pareto front size: 9
Running SPEA2 — 100 evaluations...


[2026-03-24 02:29:39,805] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:39,805] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-03-24 02:29:39,805] [jmetal.core.algorithm] [DEBUG] Finished!
[2026-03-24 02:29:39,806] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:39,807] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-24 02:29:39,928] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:39,929] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-03-24 02:29:39,929] [jmetal.core.algorithm] [DEBUG] Finished!
[2026-03-24 02:29:39,930] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-24 02:29:39,930] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...


Done. Pareto front size: 8
Running SPEA2 — 100 evaluations...
Done. Pareto front size: 5
Running SPEA2 — 100 evaluations...


[2026-03-24 02:29:40,031] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-24 02:29:40,031] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-03-24 02:29:40,032] [jmetal.core.algorithm] [DEBUG] Finished!


Done. Pareto front size: 7
Fin du protocole expérimental (tuning + runs).


# Maximisation F1

# Optimisation précision et sensibilité

# Evaluation et comparaison

# Analyse

# Comparaison avec sklearn (C4.5, RF, SVM)

# Interprétabilité